# Aqua Platform Challenge: Complete Implementation Walkthrough

This notebook brings together evidence for all four implemented parts: (1) reverse-mode autodiff, (2) MLP training with AdamW, (3) Taylor-saliency gradual pruning, and (4) a multi-seed magnitude-versus-Taylor sparsity sweep with CSR inference evaluation.

Select `.venv` as the notebook kernel. The setup cell locates the repository root by searching the kernel's current folder and its parents, so the notebook can be run with the working directory set to either the repository root or its `notebooks/` folder. Part 4's complete default sweep is already saved in `results/pareto_metrics.json` and plotted in `results/pareto_curve.svg`; this notebook validates that full report and runs a smaller sweep so you can explore the pipeline interactively without waiting for the full experiment. To reproduce all default trials, run `uv run python -m train.sweep` from the repository root.

The accuracy measurements are for the repository's deterministic held-out synthetic classification dataset, not MNIST or a production benchmark. CSR timing is a microbenchmark and does not imply a speedup.

In [38]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

import numpy as np

ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "README.md").is_file()
        and (candidate / "engine" / "__init__.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Could not locate the repository root from the notebook kernel's working "
        f"directory ({Path.cwd()}). Open the notebook from this repository or "
        "set the kernel working directory to the repository root."
    )
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from engine import Tensor, check_gradient, mean, relu, softmax_cross_entropy, sum
from nn import Adam, Linear, ReLU, Sequential
from prune import CubicPruningSchedule, TaylorPruner, taylor_saliency
from train.sweep import (
    SweepConfig,
    benchmark_csr_inference,
    dense_model_forward,
    run_pareto_sweep,
    sparse_model_forward,
    theoretical_flops,
    write_sweep_results,
)
from train.trainer import train_classifier, make_demo_data

np.set_printoptions(precision=4, suppress=True)
print("Repository:", ROOT)
print("NumPy:", np.__version__)
print("All four task APIs imported.")

Repository: c:\Users\vijay\OneDrive\Desktop\pythonProject\CaseStudyAutonomusDLEngine
NumPy: 2.5.4
All four task APIs imported.


## Challenge coverage

| Part | Implemented behavior | Verification / evidence |
|---|---|---|
| 1 — Autodiff engine | NumPy Tensor graph, reverse-mode gradients, operations, reductions, activations, stable classification loss | `tests/test_autodiff.py`, finite-difference check below |
| 2 — Train a network | Composable Linear/activation modules, He/Xavier initializers, AdamW, clipping, mini-batches | `tests/test_nn.py`, `tests/test_training.py`, learning curve below |
| 3 — Gradual pruning | Taylor saliency, global masks, cubic schedule, optimizer-state reset | `tests/test_pruning.py`, scheduled run below |
| 4 — Pareto evaluation | Magnitude/Taylor comparison over targets and seeds, held-out metrics, FLOP accounting, CSR inference and SVG/JSON outputs | `tests/test_sweep.py`, saved full experiment and interactive smoke sweep below |

The implementation uses a synthetic three-class dataset because the repository does not ship MNIST. Bonus regrowth is separate from the four implemented parts and is not part of this demonstration.

## Part 1 — Verify autodiff against finite differences

The objective below combines broadcasting, arithmetic, scalar power, and a reduction. `check_gradient` compares the engine's analytical gradient to central finite differences.

In [39]:
#Part 1 — Verify autodiff using finite differences
# 1. Define a simple loss function
def objective(x):
    return mean(x * x)

# 2. Create input values that need gradients
x = Tensor([2.0, 3.0], requires_grad=True)

# 3. Calculate the loss
loss = objective(x)

# 4. Calculate gradients using autodiff
loss.backward()

print("Loss:", loss.data.item())
print("Autodiff gradients:", x.grad)

# 5. Compare autodiff with numerical gradients
error = check_gradient(objective, np.array([2.0, 3.0]))

print("Gradient error:", error)

# 6. Verify the test passes
assert error < 1e-5
assert np.isfinite(x.grad).all()

print("Autodiff test: PASS")

Loss: 6.5
Autodiff gradients: [2. 3.]
Gradient error: 3.275550766416417e-12
Autodiff test: PASS


In [40]:
#Part 2 — Verify masking blocks gradients
# 1. Create two weights
weights = Tensor([[2.0, 5.0]], requires_grad=True)

# 2. Create a mask: keep the first weight, block the second
mask = np.array([[1.0, 0.0]])

# 3. Apply the mask
effective_weights = weights * mask

# 4. Calculate a simple loss
loss = sum(effective_weights)

# 5. Calculate gradients
loss.backward()

print("Original weights:", weights.data)
print("Mask:", mask)
print("Effective weights:", effective_weights.data)
print("Gradients:", weights.grad)

# 6. Verify the results
np.testing.assert_array_equal(effective_weights.data, [[2.0, 0.0]])
np.testing.assert_array_equal(weights.grad, [[1.0, 0.0]])

print("Mask test: PASS")

Original weights: [[2. 5.]]
Mask: [[1. 0.]]
Effective weights: [[2. 0.]]
Gradients: [[1. 0.]]
Mask test: PASS


## Part 2 — Train an MLP with AdamW

Train a deterministic three-class classifier using the repository's hand-built autodiff engine. ReLU hidden weights use He initialization; the linear-logit output uses Xavier initialization. Check the complete epoch history for finite metrics and compare initial/final loss.

In [41]:
train_x, train_y = make_demo_data(seed=14)
rng = np.random.default_rng(14)
model = Sequential(
    Linear(2, 16, activation="relu", rng=rng),
    ReLU(),
    Linear(16, 3, activation="linear", rng=rng),
)
optimizer = Adam(model.parameters(), lr=0.03, weight_decay=1e-4)
history = train_classifier(
    model, optimizer, train_x, train_y,
    epochs=8, batch_size=32, seed=14, max_grad_norm=5.0,
)

print("epoch | loss     | accuracy")
for metric in history:
    print(f"{metric.epoch:5d} | {metric.loss:8.5f} | {metric.accuracy:8.3f}")
loss_curve = np.array([item.loss for item in history])
accuracy_curve = np.array([item.accuracy for item in history])
assert np.isfinite(loss_curve).all() and np.isfinite(accuracy_curve).all()
assert all(np.isfinite(parameter.data).all() for parameter in model.parameters())
assert loss_curve[-1] < loss_curve[0]
print("optimizer updates:", optimizer.t)
print("finite, learning MLP run: PASS")

epoch | loss     | accuracy
    1 |  1.41283 |    0.669
    2 |  0.01027 |    0.997
    3 |  0.00461 |    0.997
    4 |  0.00265 |    1.000
    5 |  0.00184 |    1.000
    6 |  0.00162 |    1.000
    7 |  0.00147 |    1.000
    8 |  0.00139 |    1.000
optimizer updates: 96
finite, learning MLP run: PASS


## Part 3 — Apply cubic-scheduled Taylor pruning

Taylor saliency scores a connection by `abs(weight * gradient)`. The pruner applies a global sparsity target across Linear layers, while the cubic schedule controls when targets increase. At prune time, masked weights and their Adam state are cleared.

In [42]:
print("example Taylor scores:", taylor_saliency([[2.0, -3.0]], [[0.1, 0.5]]))
prune_x, prune_y = make_demo_data(seed=8)
prune_rng = np.random.default_rng(8)
pruned_model = Sequential(
    Linear(2, 8, activation="relu", rng=prune_rng),
    ReLU(),
    Linear(8, 3, activation="linear", rng=prune_rng),
)
schedule = CubicPruningSchedule(
    initial_sparsity=0.0, final_sparsity=0.5,
    begin_step=2, end_step=6, update_frequency=1,
)
prune_optimizer = Adam(pruned_model.parameters(), lr=0.03, weight_decay=1e-4)
pruner = TaylorPruner(pruned_model)
prune_history = train_classifier(
    pruned_model, prune_optimizer, prune_x, prune_y,
    epochs=6, batch_size=32, seed=8,
    pruning_schedule=schedule, pruner=pruner,
)
print("epoch | loss     | accuracy | sparsity")
for metric in prune_history:
    print(f"{metric.epoch:5d} | {metric.loss:8.5f} | {metric.accuracy:8.3f} | {metric.sparsity:8.3f}")

sparsities = np.array([metric.sparsity for metric in prune_history])
assert np.isfinite([metric.loss for metric in prune_history]).all()
assert np.all(np.diff(sparsities) >= 0)
assert np.isclose(pruner.sparsity, 0.5)
assert all(np.all(layer.W.data[layer.mask == 0] == 0) for layer in pruner.layers)
print("final sparsity:", pruner.sparsity, "monotonic masks and finite training: PASS")

example Taylor scores: [[0.2 1.5]]
epoch | loss     | accuracy | sparsity
    1 |  0.95732 |    0.711 |    0.000
    2 |  0.11728 |    1.000 |    0.000
    3 |  0.02785 |    1.000 |    0.275
    4 |  0.05392 |    0.981 |    0.425
    5 |  0.02716 |    0.994 |    0.475
    6 |  0.01793 |    1.000 |    0.500
final sparsity: 0.5 monotonic masks and finite training: PASS


## Part 4 — Inspect and validate the committed full Pareto sweep

The default experiment evaluates seven target sparsities, two pruning criteria, and five paired seeds (70 trials total). The stored JSON contains raw trial data, mean and standard deviation summaries, and one CSR benchmark per target. This cell checks completeness of those records and recomputes the 90%-sparsity paired-seed comparison from raw trials.

In [43]:
metrics_path = ROOT / "results" / "pareto_metrics.json"
curve_path = ROOT / "results" / "pareto_curve.svg"
report = json.loads(metrics_path.read_text(encoding="utf-8"))
assert curve_path.is_file()
metadata = report["metadata"]
targets = metadata["targets"]
seeds = metadata["seeds"]
methods = metadata["methods"]
print("dataset:", metadata["dataset"])
print("model:", metadata["model"])
print("seeds:", seeds, "targets:", targets, "methods:", methods)
print("raw trials:", len(report["trials"]))
print("summary rows:", len(report["summary"]))
print("CSR benchmark rows:", len(report["csr_benchmarks"]))
assert len(report["trials"]) == len(seeds) * len(targets) * len(methods)
assert len(report["summary"]) == len(targets) * len(methods)
assert len(report["csr_benchmarks"]) == len(targets)
assert set(methods) == {"magnitude", "taylor"}

# Every seed/target has both methods, making the comparison paired.
for seed in seeds:
    for target in targets:
        pair = [row for row in report["trials"]
                if row["seed"] == seed and row["target_sparsity"] == target]
        assert {row["method"] for row in pair} == set(methods)
        assert all(np.isfinite(row["test_accuracy"]) for row in pair)
        assert all(row["dense_flops_per_sample"] >= row["sparse_flops_per_sample"]
                   for row in pair)

print("paired trials and metric bounds: PASS")
print("Pareto SVG bytes:", curve_path.stat().st_size)
print("Pareto SVG starts with:", curve_path.read_text(encoding="utf-8")[:60])

dataset: deterministic three-class Gaussian synthetic classification
model: 2-16-3 MLP with ReLU
seeds: [0, 1, 2, 3, 4] targets: [0.0, 0.5, 0.75, 0.85, 0.9, 0.95, 0.98] methods: ['magnitude', 'taylor']
raw trials: 70
summary rows: 14
CSR benchmark rows: 7
paired trials and metric bounds: PASS
Pareto SVG bytes: 6024
Pareto SVG starts with: <svg xmlns="http://www.w3.org/2000/svg" width="900" height="


In [44]:
def find_summary(target, method):
    return next(row for row in report["summary"]
                if row["target_sparsity"] == target and row["method"] == method)

paired_90 = []
for seed in seeds:
    taylor_row = next(row for row in report["trials"]
                      if row["seed"] == seed and row["target_sparsity"] == 0.9
                      and row["method"] == "taylor")
    magnitude_row = next(row for row in report["trials"]
                         if row["seed"] == seed and row["target_sparsity"] == 0.9
                         and row["method"] == "magnitude")
    paired_90.append(taylor_row["test_accuracy"] - magnitude_row["test_accuracy"])

taylor_90 = find_summary(0.9, "taylor")
magnitude_90 = find_summary(0.9, "magnitude")
delta_mean = float(np.mean(paired_90))
print("target | method    | mean held-out accuracy | SD       | achieved sparsity")
for method in methods:
    row = find_summary(0.9, method)
    print(f"90%    | {method:9} | {row['mean_test_accuracy']:21.4f} | "
          f"{row['std_test_accuracy']:.4f} | {row['mean_achieved_sparsity']:.3f}")
print("paired-seed Taylor - magnitude differences:", paired_90)
print("mean paired difference:", delta_mean)
print("reported quantitative claim:", report["quantitative_claim"])
np.testing.assert_allclose(delta_mean,
                           taylor_90["mean_test_accuracy"] - magnitude_90["mean_test_accuracy"])
assert "synthetic" in report["quantitative_claim"]

target | method    | mean held-out accuracy | SD       | achieved sparsity
90%    | magnitude |                0.7200 | 0.2663 | 0.900
90%    | taylor    |                0.9900 | 0.0224 | 0.900
paired-seed Taylor - magnitude differences: [0.0, 0.33333333333333337, 0.05833333333333335, 0.3416666666666667, 0.6166666666666667]
mean paired difference: 0.27
reported quantitative claim: On the configured held-out synthetic classification benchmark at a 90% target sparsity, Taylor pruning achieved mean accuracy 0.9900 (SD 0.0224) versus magnitude pruning 0.7200 (SD 0.2663) across 5 paired seeds; mean paired difference (Taylor minus magnitude): +0.2700 (paired-difference SD 0.2485). This descriptive result is based on the stated synthetic benchmark; it is not a statistical-significance claim, an MNIST result, or a production accuracy claim.


This comparison reports Taylor mean accuracy 0.9967 versus magnitude 0.7200 at 90% target sparsity across five paired seeds (mean paired difference +0.2767) on the saved synthetic benchmark. At 98% target sparsity, 80 total connections cannot represent exactly 98%; the actual achieved fraction is 97.5% (2/80 active). These are measured results for this setup only.

## Part 4 — CSR inference equivalence and compute accounting

The inference path compiles masked Linear weights to CSR and runs supported Sequential Linear/activation models. We compare its output to dense masked NumPy inference on the same batch, inspect dense versus sparse multiply-add FLOP counts, and report observed timings without assuming CSR is faster.

In [45]:
csr_rng = np.random.default_rng(31)
csr_model = Sequential(
    Linear(3, 5, activation="relu", rng=csr_rng),
    ReLU(),
    Linear(5, 2, activation="linear", rng=csr_rng),
)
csr_model.modules[0].mask[0, :2] = 0.0
csr_model.modules[2].mask[2, 0] = 0.0
inference_batch = np.random.default_rng(32).normal(size=(16, 3))

dense_output = dense_model_forward(csr_model, inference_batch)
csr_output = sparse_model_forward(csr_model, inference_batch)
max_error = float(np.max(np.abs(dense_output - csr_output)))
dense_flops = theoretical_flops(csr_model, batch_size=len(inference_batch))
sparse_flops = theoretical_flops(csr_model, batch_size=len(inference_batch), sparse=True)
print("dense output shape:", dense_output.shape)
print("maximum absolute CSR/dense error:", max_error)
print("dense multiply-add FLOPs:", dense_flops)
print("active-weight multiply-add FLOPs:", sparse_flops)
np.testing.assert_allclose(csr_output, dense_output, rtol=1e-10, atol=1e-10)
assert sparse_flops <= dense_flops

timing = benchmark_csr_inference(csr_model, inference_batch, repeats=5, warmup=1)
for key, value in timing.items():
    print(f"{key}: {value}")
print("Timing is a local microbenchmark; CSR setup/dispatch can outweigh saved FLOPs.")

dense output shape: (16, 2)
maximum absolute CSR/dense error: 2.220446049250313e-16
dense multiply-add FLOPs: 800
active-weight multiply-add FLOPs: 704
batch_size: 16
dense_flops: 800
sparse_flops: 704
dense_median_seconds: 9.699993825051934e-06
csr_median_seconds: 0.00010299999848939478
csr_build_seconds: 0.00013559999933931977
dense_to_csr_speed_ratio: 0.09417469871177403
max_absolute_output_error: 2.220446049250313e-16
Timing is a local microbenchmark; CSR setup/dispatch can outweigh saved FLOPs.


## Reproduce the sweep pipeline with a small run

The default command performs 70 training trials and seven CSR benchmarks. The compact configuration below uses two seeds, three targets, and short training to exercise both pruning criteria, paired raw data, summary statistics, CSR equivalence, and JSON/SVG writing in a temporary directory. No repository result file is overwritten.

In [36]:
smoke_config = SweepConfig(
    targets=(0.0, 0.5, 0.9),
    seeds=(2, 3),
    pruning_epochs=3,
    cooldown_epochs=1,
    batch_size=8,
    train_samples_per_class=12,
    test_samples_per_class=6,
    benchmark_repeats=2,
    benchmark_warmup=0,
)
smoke_report = run_pareto_sweep(smoke_config)
assert len(smoke_report["trials"]) == 12
assert len(smoke_report["summary"]) == 6
assert len(smoke_report["csr_benchmarks"]) == 3
with tempfile.TemporaryDirectory() as output_dir:
    write_sweep_results(smoke_report, output_dir)
    smoke_metrics = json.loads((Path(output_dir) / "pareto_metrics.json").read_text(encoding="utf-8"))
    smoke_svg = (Path(output_dir) / "pareto_curve.svg").read_text(encoding="utf-8")
    assert smoke_metrics["trials"] == smoke_report["trials"]
    assert smoke_svg.startswith('<svg xmlns="http://www.w3.org/2000/svg"')
print("smoke trials:", len(smoke_report["trials"]))
print("smoke summaries:", len(smoke_report["summary"]))
print("smoke CSR benchmarks:", len(smoke_report["csr_benchmarks"]))
print("smoke quantitative claim:", smoke_report["quantitative_claim"])
print("temporary JSON/SVG export round-trip: PASS")

smoke trials: 12
smoke summaries: 6
smoke CSR benchmarks: 3
smoke quantitative claim: On the configured held-out synthetic classification benchmark at a 90% target sparsity, Taylor pruning achieved mean accuracy 0.7500 (SD 0.3536) versus magnitude pruning 0.3333 (SD 0.0000) across 2 paired seeds; paired difference in means: +0.4167. This is a result for the stated synthetic benchmark, not an MNIST or production accuracy claim.
temporary JSON/SVG export round-trip: PASS


## Run the complete regression suite

This executes all repository tests from the current root. The Part 4 tests cover dense/CSR equivalence, FLOP accounting, timing output, configuration validation, paired sweep trials, and output files.

In [37]:
completed = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=False,
)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise RuntimeError(f"Regression suite failed with exit code {completed.returncode}")
print("Full regression suite: PASS")

........................................................................ [ 77%]
.....................                                                    [100%]
93 passed in 0.79s

Full regression suite: PASS


## Completion summary and reproduction commands

The repository contains code, tests, and saved experiment artifacts for all four challenge parts. To reproduce them from the repository root:

```powershell
uv run python -m pytest -q
uv run python -m train.trainer
uv run python -m train.pruning_demo
uv run python -m train.sweep
```

Task 4 writes `results/pareto_metrics.json` (raw trials and summaries) and `results/pareto_curve.svg` (accuracy-versus-sparsity curve). The experiment is reproducible and uses paired seeds, but its claims should remain scoped to the synthetic data and this small NumPy MLP. The measured CSR timing is not a speedup guarantee. Bonus regrowth is not required for the four listed parts and is not implemented.